In [64]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [65]:
import os

PROJECT_PATH = "/content/European-Energy-Markets-Dashboard"

os.makedirs(f"{PROJECT_PATH}/data/raw", exist_ok=True)
os.makedirs(f"{PROJECT_PATH}/data/processed", exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [66]:
import os

folders = [
    "/content/data/raw",
    "/content/data/processed"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created!")

Folders created!


In [67]:
prices = pd.read_csv("/content/data/raw/prices.csv")

In [68]:
prices.head()
prices.info()
prices.columns.tolist()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119203 entries, 0 to 119202
Data columns (total 4 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Country           119203 non-null  object 
 1   ISO3 Code         119203 non-null  object 
 2   Date              119203 non-null  object 
 3   Price (EUR/MWhe)  119119 non-null  float64
dtypes: float64(1), object(3)
memory usage: 3.6+ MB


['Country', 'ISO3 Code', 'Date', 'Price (EUR/MWhe)']

In [69]:
prices.isna().sum()

,0
Country,0
ISO3 Code,0
Date,0
Price (EUR/MWhe),84


In [70]:
#keep only 8 countries we want
selected_countries = [
    "Germany",
    "France",
    "Italy",
    "Spain",
    "Netherlands",
    "Belgium",
    "Poland",
    "Greece"
]

prices = prices[prices["Country"].isin(selected_countries)]

prices.shape

(33627, 4)

In [71]:
prices["Date"] = pd.to_datetime(prices["Date"])
prices.info()

<class 'pandas.core.frame.DataFrame'>
Index: 33627 entries, 1 to 119200
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Country           33627 non-null  object        
 1   ISO3 Code         33627 non-null  object        
 2   Date              33627 non-null  datetime64[ns]
 3   Price (EUR/MWhe)  33625 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 1.3+ MB


In [72]:
#rename columns
prices.rename(columns={
    "ISO3 Code": "ISO3",
    "Price (EUR/MWhe)": "Price"
}, inplace=True)

prices.head()

,Country,ISO3,Date,Price
1,Belgium,BEL,2015-01-01,36.56
6,France,FRA,2015-01-01,36.56
7,Germany,DEU,2015-01-01,22.34
8,Greece,GRC,2015-01-01,43.37
10,Italy,ITA,2015-01-01,47.72


In [73]:
prices = prices.sort_values(["Country", "Date"])

In [74]:
prices.to_csv(
    "/content/data/processed/prices_clean.csv",
    index=False
)

In [75]:
#Load Generation Dataset
generation_url = "https://storage.googleapis.com/emb-prod-bkt-publicdata/public-downloads/monthly_full_release_long_format.csv"

generation = pd.read_csv(generation_url)

print("Dataset loaded!")

Dataset loaded!


In [76]:
generation.head()

,Area,ISO 3 code,Date,Area type,Continent,Ember region,EU,OECD,G20,G7,ASEAN,Category,Subcategory,Variable,Unit,Value,YoY absolute change,YoY % change
0,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity demand,Demand,Demand,TWh,12.77,NaN,NaN
1,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,Clean,%,34.57,NaN,NaN
2,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,Fossil,%,65.44,NaN,NaN
3,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,Gas and Other Fossil,%,63.40,NaN,NaN
4,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,"Hydro, Bioenergy and Other Renewables",%,29.08,NaN,NaN


In [77]:
generation.columns.tolist()

['Area',
 'ISO 3 code',
 'Date',
 'Area type',
 'Continent',
 'Ember region',
 'EU',
 'OECD',
 'G20',
 'G7',
 'ASEAN',
 'Category',
 'Subcategory',
 'Variable',
 'Unit',
 'Value',
 'YoY absolute change',
 'YoY % change']

In [78]:
generation["Category"].unique()

array(['Electricity demand', 'Electricity generation',
       'Electricity imports', 'Power sector emissions',
       'Electricity prices'], dtype=object)

In [79]:
generation["Variable"].unique()

array(['Demand', 'Clean', 'Fossil', 'Gas and Other Fossil',
       'Hydro, Bioenergy and Other Renewables', 'Renewables',
       'Wind and Solar', 'Bioenergy', 'Coal', 'Gas', 'Hydro', 'Nuclear',
       'Other Fossil', 'Solar', 'Wind', 'Total Generation', 'Net Imports',
       'CO2 intensity', 'Total emissions', 'Other Renewables',
       'Day-ahead electricity price'], dtype=object)

In [80]:
selected_countries = [
    "Germany",
    "France",
    "Italy",
    "Spain",
    "Netherlands",
    "Belgium",
    "Poland",
    "Greece"
]

generation = generation[
    generation["Area"].isin(selected_countries)
]

generation.shape

(52375, 18)

In [81]:
generation["Date"] = pd.to_datetime(generation["Date"])

In [82]:
keep_variables = [
    "Demand",
    "Wind",
    "Solar",
    "Hydro",
    "Bioenergy",
    "Coal",
    "Gas",
    "Nuclear",
    "Net Imports",
    "Total Generation",
    "Renewables"
]


generation = generation[
    generation["Variable"].isin(keep_variables)
]

In [83]:
generation_clean = generation[
    [
        "Area",
        "ISO 3 code",
        "Date",
        "Variable",
        "Unit",
        "Value"
    ]
]

In [84]:
generation_clean.rename(
    columns={
        "Area":"Country",
        "ISO 3 code":"ISO3",
        "Variable":"Metric"
    },
    inplace=True
)

generation_clean.head()

/tmp/ipykernel_1410/1682437813.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  generation_clean.rename(


,Country,ISO3,Date,Metric,Unit,Value
50062,Belgium,BEL,2015-01-01,Demand,TWh,8.04
50067,Belgium,BEL,2015-01-01,Renewables,%,13.02
50073,Belgium,BEL,2015-01-01,Renewables,TWh,0.88
50075,Belgium,BEL,2015-01-01,Bioenergy,%,2.96
50076,Belgium,BEL,2015-01-01,Coal,%,3.70


In [85]:
generation_clean["Metric"].value_counts()

,count
Metric,
Renewables,3252
Gas,3252
Coal,3252
Wind,3252
Solar,3252
Hydro,3252
Bioenergy,2841
Nuclear,2055
Demand,1084


In [86]:
generation_clean.to_csv(
    "/content/data/processed/generation_clean.csv",
    index=False
)

In [87]:
generation_clean = generation_clean[
    (generation_clean["Date"] >= "2022-01-01") &
    (generation_clean["Date"] <= "2024-12-31")
]

generation_clean.shape

(7344, 6)

In [88]:
prices_monthly = (
    prices
    .groupby(
        [
            "Country",
            "ISO3",
            prices["Date"].dt.to_period("M")
        ]
    )
    ["Price"]
    .mean()
    .reset_index()
)

In [89]:
prices_monthly["Date"] = (
    prices_monthly["Date"]
    .dt
    .to_timestamp()
)

In [90]:
generation_pivot = generation_clean.pivot_table(
    index=[
        "Country",
        "ISO3",
        "Date"
    ],
    columns="Metric",
    values="Value"
).reset_index()

In [91]:
energy = prices_monthly.merge(
    generation_pivot,
    on=[
        "Country",
        "ISO3",
        "Date"
    ],
    how="left"
)

In [92]:
energy.head()

,Country,ISO3,Date,Price,Bioenergy,Coal,Demand,Gas,Hydro,Net Imports,Nuclear,Renewables,Solar,Total Generation,Wind
0,Belgium,BEL,2015-01-01,42.334839,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Belgium,BEL,2015-02-01,50.542500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Belgium,BEL,2015-03-01,47.049355,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Belgium,BEL,2015-04-01,47.689667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Belgium,BEL,2015-05-01,37.577419,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [93]:
energy.to_csv(
    "/content/data/processed/energy_fact.csv",
    index=False
)

In [94]:
prices = prices[
    (prices["Date"] >= "2022-01-01") &
    (prices["Date"] <= "2024-12-31")
]

prices.shape

(8768, 4)

In [95]:
prices_monthly = (
    prices
    .groupby(
        [
            "Country",
            "ISO3",
            prices["Date"].dt.to_period("M")
        ]
    )
    ["Price"]
    .mean()
    .reset_index()
)

prices_monthly["Date"] = prices_monthly["Date"].dt.to_timestamp()

prices_monthly.head()

,Country,ISO3,Date,Price
0,Belgium,BEL,2022-01-01,191.562258
1,Belgium,BEL,2022-02-01,162.637857
2,Belgium,BEL,2022-03-01,265.513226
3,Belgium,BEL,2022-04-01,186.718000
4,Belgium,BEL,2022-05-01,176.665806


In [96]:
print(prices_monthly["Date"].min())
print(prices_monthly["Date"].max())

print(generation_pivot["Date"].min())
print(generation_pivot["Date"].max())

2022-01-01 00:00:00
2024-12-01 00:00:00
2022-01-01 00:00:00
2024-12-01 00:00:00


In [97]:
energy = prices_monthly.merge(
    generation_pivot,
    on=[
        "Country",
        "ISO3",
        "Date"
    ],
    how="left"
)

In [98]:
energy.isna().sum()

,0
Country,0
ISO3,0
Date,0
Price,0
Bioenergy,36
Coal,0
Demand,0
Gas,0
Hydro,0
Net Imports,0


In [99]:
energy.fillna(
    {
        "Bioenergy": 0,
        "Nuclear": 0
    },
    inplace=True
)

In [100]:
energy.isna().sum()

,0
Country,0
ISO3,0
Date,0
Price,0
Bioenergy,0
Coal,0
Demand,0
Gas,0
Hydro,0
Net Imports,0


In [101]:
#Renewable Share
energy["Renewable Share %"] = (
    (
        energy["Wind"] +
        energy["Solar"] +
        energy["Hydro"] +
        energy["Bioenergy"]
    )
    /
    energy["Total Generation"]
) * 100

In [102]:
energy[
    [
        "Country",
        "Date",
        "Renewable Share %",
        "Price"
    ]
].head()

,Country,Date,Renewable Share %,Price
0,Belgium,2022-01-01,64.480245,191.562258
1,Belgium,2022-02-01,128.584110,162.637857
2,Belgium,2022-03-01,97.383852,265.513226
3,Belgium,2022-04-01,127.197802,186.718000
4,Belgium,2022-05-01,119.027583,176.665806


In [103]:
energy.to_csv(
    "/content/data/processed/energy_fact.csv",
    index=False
)

In [104]:
energy.head()

,Country,ISO3,Date,Price,Bioenergy,Coal,Demand,Gas,Hydro,Net Imports,Nuclear,Renewables,Solar,Total Generation,Wind,Renewable Share %
0,Belgium,BEL,2022-01-01,191.562258,0.806667,0.0,7.99,9.310000,0.000000,-0.70,18.516667,5.603333,0.416667,8.69,4.380000,64.480245
1,Belgium,BEL,2022-02-01,162.637857,0.483333,0.0,7.03,5.700000,0.000000,-0.48,17.520000,9.656667,1.053333,7.51,8.120000,128.584110
2,Belgium,BEL,2022-03-01,265.513226,0.886667,0.0,7.23,7.776667,0.096667,-0.16,17.640000,7.196667,2.723333,7.39,3.490000,97.383852
3,Belgium,BEL,2022-04-01,186.718000,0.850000,0.0,6.83,5.610000,0.096667,-0.45,17.936667,9.260000,3.496667,7.28,4.816667,127.197802
4,Belgium,BEL,2022-05-01,176.665806,0.506667,0.0,6.87,6.640000,0.050000,-0.26,17.986667,8.486667,4.470000,7.13,3.460000,119.027583


In [105]:
energy.drop(
    columns=["Renewable Share %"],
    inplace=True
)

In [106]:
energy.head()

,Country,ISO3,Date,Price,Bioenergy,Coal,Demand,Gas,Hydro,Net Imports,Nuclear,Renewables,Solar,Total Generation,Wind
0,Belgium,BEL,2022-01-01,191.562258,0.806667,0.0,7.99,9.310000,0.000000,-0.70,18.516667,5.603333,0.416667,8.69,4.380000
1,Belgium,BEL,2022-02-01,162.637857,0.483333,0.0,7.03,5.700000,0.000000,-0.48,17.520000,9.656667,1.053333,7.51,8.120000
2,Belgium,BEL,2022-03-01,265.513226,0.886667,0.0,7.23,7.776667,0.096667,-0.16,17.640000,7.196667,2.723333,7.39,3.490000
3,Belgium,BEL,2022-04-01,186.718000,0.850000,0.0,6.83,5.610000,0.096667,-0.45,17.936667,9.260000,3.496667,7.28,4.816667
4,Belgium,BEL,2022-05-01,176.665806,0.506667,0.0,6.87,6.640000,0.050000,-0.26,17.986667,8.486667,4.470000,7.13,3.460000


In [107]:
energy.to_csv(
    "/content/data/processed/energy_fact.csv",
    index=False
)

In [109]:
#Year / Month columns
energy["Year"] = energy["Date"].dt.year
energy["Month"] = energy["Date"].dt.month
energy["Month Name"] = energy["Date"].dt.month_name()

energy.duplicated().sum()

np.int64(0)

In [110]:
energy.to_csv(
    "/content/data/processed/energy_fact_final.csv",
    index=False
)